In [1]:
import subprocess
subprocess.run(['pip', 'install', 'pymongo'], capture_output=True)

CompletedProcess(args=['pip', 'install', 'pymongo'], returncode=0, stdout=b'Collecting pymongo\n  Downloading pymongo-4.17.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)\nCollecting dnspython<3.0.0,>=2.6.1 (from pymongo)\n  Downloading dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)\nDownloading pymongo-4.17.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.5 MB)\n\x1b[?25l   \x1b\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\x1b \x1b0.0/1.5 MB\x1b \x1b?\x1b eta \x1b-:-

In [2]:

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("LoadToMongoDB") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

user_features = spark.read.parquet('/home/jovyan/data/features/user_features')
product_features = spark.read.parquet('/home/jovyan/data/features/product_features')
up_features = spark.read.parquet('/home/jovyan/data/features/up_features')


In [4]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient('mongodb://instacart-mongodb:27017/')
db = client['instacart']

# Load and insert user_features
user_pd = user_features.toPandas()
db['user_features'].drop()
db['user_features'].insert_many(user_pd.to_dict('records'))
print(f'Inserted {len(user_pd)} user features')

product_pd = product_features.toPandas()
db['product_features'].drop()
db['product_features'].insert_many(product_pd.to_dict('records'))
print(f'Inserted {len(product_pd)} product features')

up_pd = up_features.toPandas()
db['up_features'].drop()
db['up_features'].insert_many(up_pd.to_dict('records'))
print(f'Inserted {len(up_pd)} up features')

Inserted 206209 user features
Inserted 43713 product features
Inserted 2779603 up features


In [ ]:
client = MongoClient('mongodb://instacart-mongodb:27017/')
db = client['instacart']

print('user_features:', db['user_features'].count_documents({}))
print('product_features:', db['product_features'].count_documents({}))
print('up_features:', db['up_features'].count_documents({}))

user_features: 206209
product_features: 43713
up_features: 2779603
